# 01 · Data Preprocessing — COSMO belief–affect–behavior network pipeline

This notebook applies fixed preprocessing rules (established in `00_data_audit`)
and constructs the complete-case analysis datasets used for network estimation.

### Goals
1. Restrict the dataset to the waves selected by the audit (`WAVES_SELECTED = [55, 56, 57]`).
2. Apply all codebook-based recoding rules in a single, ordered pass:
   - Generic missing-value sentinel codes → `NaN`.
   - `TRUST_RKI` non-substantive codes (`1`, `2`) → `NaN` *(consistent with audit)*.
   - `USE2_*` category `1` ("Trifft nicht zu") → `NaN`.
   - Reverse-code `AFF_FEAR` and `AFF_WORRY` (formula: `8 − x`) so that  
     higher values consistently reflect more negative affect / greater perceived threat.
3. Keep behavior variables as **ordinal frequency** measures (levels 2–6).
4. Construct **trust strata** (low / high) within each wave using `TRUST_RKI`:
   - Bottom tercile → `"low"`, top tercile → `"high"`, middle tercile excluded.
   - Tercile boundaries are computed on the non-missing `TRUST_RKI` values within  
     each wave separately (wave-specific percentiles).
5. Export one complete-case CSV per network instance (`wave × trust stratum`) and  
   save an instance index for downstream estimation scripts.

### ⚠ Consistency notes
- Recoding rules here **must match** `00_data_audit` exactly (same sentinel codes,  
  same `TRUST_RKI` non-substantive values, same `USE2_*` category 1 rule).
- `WAVES_SELECTED` is set manually from the audit output; do **not** rerun the  
  wave-selection logic here.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH   = Path("..") / "data" / "raw_data_COSMO.csv"
OUTPUT_DIR  = Path("..") / "data" / "instances"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print("Loaded df shape :", df.shape)
print("Number of cols  :", len(df.columns))

Loaded df shape : (70010, 334)
Number of cols  : 334


In [2]:
# ── Wave column ───────────────────────────────────────────────────────────────
WAVE = "TIME"
if WAVE not in df.columns:
    raise ValueError(f"'{WAVE}' not found in df.columns")
df[WAVE] = pd.to_numeric(df[WAVE], errors="coerce")

# ── Variable lists (must match 00_data_audit exactly) ─────────────────────────
FOCAL_BELIEF  = "SEVERITY"
AFFECT_VARS   = ["AFF_FEAR", "AFF_WORRY", "WORRY_HEALTH_SYSTEM"]
BEHAVIOR_VARS = ["USE2_MASK", "USE2_SPACE150", "USE2_HANDWASH20", "USE2_AVOID"]
TRUST_VARS    = ["TRUST_RKI"]   # context descriptor — not a network node

NODE_VARS  = [FOCAL_BELIEF] + AFFECT_VARS + BEHAVIOR_VARS
VARIABLES  = NODE_VARS + TRUST_VARS

# ── Waves selected by audit ────────────────────────────────────────────────────
# Set manually from the output of 00_data_audit; do not recompute here.
WAVES_SELECTED = [55, 56, 57]

# ── Verify all required columns are present ────────────────────────────────────
missing_cols = [c for c in [WAVE] + VARIABLES if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("NODE_VARS        :", NODE_VARS)
print("TRUST_VARS       :", TRUST_VARS)
print("WAVES_SELECTED   :", WAVES_SELECTED)

NODE_VARS        : ['SEVERITY', 'AFF_FEAR', 'AFF_WORRY', 'WORRY_HEALTH_SYSTEM', 'USE2_MASK', 'USE2_SPACE150', 'USE2_HANDWASH20', 'USE2_AVOID']
TRUST_VARS       : ['TRUST_RKI']
WAVES_SELECTED   : [55, 56, 57]


In [3]:
# ── RECODING — single ordered pass ───────────────────────────────────────────
# All transformations are applied here, before any subsetting, so that
# the recoded state is unambiguous at every downstream step.

# ── Step 1 · Restrict to selected waves ──────────────────────────────────────
df = df.loc[df[WAVE].isin(WAVES_SELECTED), [WAVE] + VARIABLES].copy()
print(f"After wave restriction: {df.shape}  waves present: "
      f"{sorted(df[WAVE].dropna().unique().tolist())}")

# ── Step 2 · Ensure numeric dtype ────────────────────────────────────────────
for v in VARIABLES:
    df[v] = pd.to_numeric(df[v], errors="coerce")

# ── Step 3 · Generic sentinel codes → NaN  (must match 00_data_audit) ─────────
MISSING_CODES = [
    -99, -98, -97, -96, -95, -94, -93, -92, -91, -90,
    -9,  -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,
    95, 96, 97, 98, 99,
    999, 9999,
]
df[VARIABLES] = df[VARIABLES].replace(MISSING_CODES, np.nan)

# ── Step 4 · Codebook-specific non-substantive recodes ────────────────────────
# TRUST_RKI: 1 = "keine Angabe möglich", 2 non-substantive (see audit) → NaN
_trust_nonsub = [1, 2]
df.loc[df["TRUST_RKI"].isin(_trust_nonsub), "TRUST_RKI"] = np.nan

# USE2_* behavior: 1 = "Trifft nicht zu" (not applicable) → NaN
for v in BEHAVIOR_VARS:
    df.loc[df[v].eq(1), v] = np.nan

# ── Step 5 · Reverse-code affect items (1-7 scale; higher = more threat) ──────
# Formula: 8 − x  maps  1→7, 2→6, … 7→1; NaN arithmetic preserves NaN.
# ⚠ Apply ONLY ONCE — do not repeat in later cells.
AFF_REVERSE = ["AFF_FEAR", "AFF_WORRY"]
for v in AFF_REVERSE:
    df[v] = 8 - df[v]

# ── Confirmation printout ──────────────────────────────────────────────────────
print("\nPost-recode value ranges:")
EXPECTED_POST = {
    "SEVERITY"           : (1, 7),
    "AFF_FEAR"           : (1, 7),   # after reversal: 8-7=1 … 8-1=7
    "AFF_WORRY"          : (1, 7),
    "WORRY_HEALTH_SYSTEM": (1, 7),
    "TRUST_RKI"          : (3, 9),   # substantive range
    "USE2_MASK"          : (2, 6),
    "USE2_SPACE150"      : (2, 6),
    "USE2_HANDWASH20"    : (2, 6),
    "USE2_AVOID"         : (2, 6),
}
flag_any = False
for v, (lo, hi) in EXPECTED_POST.items():
    x    = df[v].dropna()
    omin = float(x.min()) if len(x) else float("nan")
    omax = float(x.max()) if len(x) else float("nan")
    out_of_range = (x < lo).any() or (x > hi).any()
    flag = " ⚠ OUT OF RANGE" if out_of_range else ""
    print(f"  {v:<22}: min={omin:.0f}  max={omax:.0f}  "
          f"unique={sorted(x.unique().tolist()[:10])}{flag}")
    if out_of_range:
        flag_any = True

if flag_any:
    raise ValueError("One or more variables have out-of-range values after recoding. "
                     "Review the output above before proceeding.")

After wave restriction: (2991, 10)  waves present: [55, 56, 57]

Post-recode value ranges:
  SEVERITY              : min=1  max=7  unique=[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
  AFF_FEAR              : min=1  max=7  unique=[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
  AFF_WORRY             : min=1  max=7  unique=[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
  WORRY_HEALTH_SYSTEM   : min=1  max=7  unique=[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
  TRUST_RKI             : min=3  max=9  unique=[3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]
  USE2_MASK             : min=2  max=6  unique=[2.0, 3.0, 4.0, 5.0, 6.0]
  USE2_SPACE150         : min=2  max=6  unique=[2.0, 3.0, 4.0, 5.0, 6.0]
  USE2_HANDWASH20       : min=2  max=6  unique=[2.0, 3.0, 4.0, 5.0, 6.0]
  USE2_AVOID            : min=2  max=6  unique=[2.0, 3.0, 4.0, 5.0, 6.0]


## Trust strata construction

Within each wave, respondents are divided into **low-trust** and **high-trust** groups  
using the bottom and top terciles of `TRUST_RKI`, computed on non-missing values.  
The middle tercile is excluded to maximize structural contrast between strata.

**Implementation notes:**
- Tercile boundaries are wave-specific (applied via `groupby`).
- Respondents with `TRUST_RKI = NaN` (after recoding) receive no stratum label.
- If the 33rd and 67th percentile values are identical (possible on a discrete scale),  
  a `ValueError` is raised so the analyst can inspect the distribution instead of  
  silently producing near-empty strata.
- Boundary inclusivity: `≤ q₃₃` → `"low"`;  `≥ q₆₇` → `"high"`.  
  This is more robust to ties than strict inequality on a 7-point discrete scale.

In [4]:
# ── Trust strata assignment (bottom / top tercile per wave) ──────────────────

TRUST = "TRUST_RKI"

def assign_trust_tercile(s: pd.Series, wave_label=None) -> pd.Series:
    """
    Assign 'low' (≤ 33rd pctile) / 'high' (≥ 67th pctile) labels within a wave.
    Middle tercile and NaN → NaN (excluded).

    Boundary: inclusive on both sides (≤ / ≥) so that ties at the boundary
    are assigned to the extreme group rather than silently dropped.
    This is defensible for a discrete 7-point trust scale.

    Raises ValueError if q33 == q67 (degenerate distribution), which would
    make stratum membership ill-defined.
    """
    s     = pd.to_numeric(s, errors="coerce")
    out   = pd.Series(np.nan, index=s.index, dtype=object)
    valid = s.dropna()

    if valid.empty:
        return out

    q33 = valid.quantile(1 / 3)
    q67 = valid.quantile(2 / 3)

    if q33 == q67:
        label = f"wave {wave_label}" if wave_label is not None else "unknown wave"
        raise ValueError(
            f"TRUST_RKI tercile boundaries are equal (q33 = q67 = {q33}) in {label}. "
            "The trust distribution is too concentrated to form meaningful strata. "
            "Inspect the distribution and consider alternative stratification."
        )

    out.loc[s <= q33] = "low"
    out.loc[s >= q67] = "high"
    return out

# Apply wave-by-wave (so boundaries are wave-specific)
strata_parts = []
for w, grp in df.groupby(WAVE):
    s_labeled = assign_trust_tercile(grp[TRUST], wave_label=int(w))
    strata_parts.append(s_labeled)

df["trust_stratum"] = pd.concat(strata_parts).reindex(df.index)

print("Trust strata counts by wave (NaN = middle tercile or missing trust):")
display(
    df.assign(trust_stratum=df["trust_stratum"].fillna("excluded_or_missing"))
    .groupby([WAVE, "trust_stratum"]).size().unstack(fill_value=0)
)

Trust strata counts by wave (NaN = middle tercile or missing trust):


trust_stratum,excluded_or_missing,high,low
TIME,,,
55,208,336,423
56,223,360,431
57,198,418,394


In [5]:
# ── Export one complete-case CSV per network instance ─────────────────────────
# Naming convention: cosmo_nodes_wave{w}_trust{g}.csv
# n_cc is stored in the index only (not in the filename) to avoid
# fragile filename-dependent lookups in validation and downstream scripts.

instances = []
for w in sorted(df[WAVE].dropna().unique()):
    for g in ["low", "high"]:
        mask  = df[WAVE].eq(w) & df["trust_stratum"].eq(g)
        d_sub = df.loc[mask, [WAVE] + NODE_VARS + TRUST_VARS].copy()
        d_cc  = d_sub.dropna(subset=NODE_VARS)

        n_cc  = int(len(d_cc))
        fname = f"cosmo_nodes_wave{int(w)}_trust{g}.csv"
        d_cc.to_csv(OUTPUT_DIR / fname, index=False)

        instances.append({
            "wave"         : int(w),
            "trust_stratum": g,
            "n_cc"         : n_cc,
            "n_stratum"    : int(mask.sum()),   # before complete-case filter
            "filename"     : fname,
        })

instances_df = (
    pd.DataFrame(instances)
    .sort_values(["wave", "trust_stratum"])
    .reset_index(drop=True)
)

index_path = OUTPUT_DIR / "instances_index.csv"
instances_df.to_csv(index_path, index=False)

print(f"Instance CSVs saved to : {OUTPUT_DIR}")
print(f"Instance index saved to: {index_path}")
print()
display(instances_df)

Instance CSVs saved to : ..\data\instances
Instance index saved to: ..\data\instances\instances_index.csv



,wave,trust_stratum,n_cc,n_stratum,filename
0,55,high,309,336,cosmo_nodes_wave55_trusthigh.csv
1,55,low,376,423,cosmo_nodes_wave55_trustlow.csv
2,56,high,351,360,cosmo_nodes_wave56_trusthigh.csv
3,56,low,383,431,cosmo_nodes_wave56_trustlow.csv
4,57,high,407,418,cosmo_nodes_wave57_trusthigh.csv
5,57,low,336,394,cosmo_nodes_wave57_trustlow.csv


In [6]:
# ── Validation checks ─────────────────────────────────────────────────────────
# Run after Cell 6 to confirm preprocessing + export correctness.

import sys

errors   = []
warnings = []

# ── 1. Wave coverage ──────────────────────────────────────────────────────────
waves_present = sorted(df[WAVE].dropna().unique().tolist())
if set(waves_present) != set(WAVES_SELECTED):
    errors.append(f"Wave mismatch: present={waves_present}, expected={WAVES_SELECTED}")
else:
    print(f"✓ Wave coverage OK: {waves_present}")

# ── 2. Post-recode value ranges ───────────────────────────────────────────────
EXPECTED_POST = {
    "SEVERITY"           : (1, 7),
    "AFF_FEAR"           : (1, 7),
    "AFF_WORRY"          : (1, 7),
    "WORRY_HEALTH_SYSTEM": (1, 7),
    "TRUST_RKI"          : (3, 9),
    "USE2_MASK"          : (2, 6),
    "USE2_SPACE150"      : (2, 6),
    "USE2_HANDWASH20"    : (2, 6),
    "USE2_AVOID"         : (2, 6),
}
for v, (lo, hi) in EXPECTED_POST.items():
    x = df[v].dropna()
    if len(x) == 0:
        warnings.append(f"{v}: no non-missing values")
        continue
    if (x < lo).any() or (x > hi).any():
        bad = sorted(x[(x < lo) | (x > hi)].unique().tolist())
        errors.append(f"{v}: out-of-range values {bad} (expected {lo}–{hi})")

if not any(v.startswith("USE2") for v in EXPECTED_POST):
    pass
# also check no 1s remain in behavior (would indicate recode failure)
for v in BEHAVIOR_VARS:
    if df[v].eq(1).any():
        errors.append(f"{v}: category 1 ('Trifft nicht zu') still present after recode")

# ── 3. Reverse-coding direction check ─────────────────────────────────────────
# After reversal: original 1 → 7, original 7 → 1.
# We cannot verify against original values here, but we can check that
# AFF_FEAR and AFF_WORRY have the same range as SEVERITY (1–7).
for v in AFF_REVERSE:
    x = df[v].dropna()
    if len(x) > 0 and (x.min() < 1 or x.max() > 7):
        errors.append(f"{v}: range outside 1–7 after reverse-coding")
    else:
        print(f"✓ {v} range OK after reversal: [{x.min():.0f}, {x.max():.0f}]")

# ── 4. Trust strata sanity ────────────────────────────────────────────────────
valid_strata = {"low", "high", np.nan}
unexpected_strata = set(df["trust_stratum"].dropna().unique()) - {"low", "high"}
if unexpected_strata:
    errors.append(f"Unexpected trust_stratum values: {unexpected_strata}")
else:
    print("✓ trust_stratum values OK: only 'low', 'high', NaN")

# ── 5. Instance index vs. exported files ──────────────────────────────────────
idx = pd.read_csv(OUTPUT_DIR / "instances_index.csv")
missing_files = []
for _, r in idx.iterrows():
    fpath = OUTPUT_DIR / r["filename"]
    if not fpath.exists():
        missing_files.append(r["filename"])

if missing_files:
    errors.append(f"Missing instance files: {missing_files}")
else:
    print(f"✓ All {len(idx)} instance files present on disk")

# ── 6. Node completeness per instance (zero missing on NODE_VARS required) ────
completeness_rows = []
for _, r in idx.iterrows():
    d_inst = pd.read_csv(OUTPUT_DIR / r["filename"])
    n_rows  = len(d_inst)
    n_miss  = int(d_inst[NODE_VARS].isna().sum().sum())
    n_idx   = int(r["n_cc"])
    row_ok  = (n_miss == 0) and (n_rows == n_idx)
    completeness_rows.append({
        "file"                   : r["filename"],
        "rows_on_disk"           : n_rows,
        "n_cc_in_index"          : n_idx,
        "total_missing_node_vals": n_miss,
        "consistent"             : row_ok,
    })
    if not row_ok:
        errors.append(f"{r['filename']}: n_rows={n_rows}, n_cc_index={n_idx}, missing={n_miss}")

comp_df = pd.DataFrame(completeness_rows)
display(comp_df)
if comp_df["consistent"].all():
    print("✓ All instance files are complete-case and consistent with the index")

# ── 7. Minimum N warning ──────────────────────────────────────────────────────
MIN_N_WARN = 100
small = idx[idx["n_cc"] < MIN_N_WARN]
if not small.empty:
    warnings.append(f"Instance(s) with n_cc < {MIN_N_WARN}: {small[['wave','trust_stratum','n_cc']].to_dict('records')}")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
if errors:
    print(f"❌ {len(errors)} ERROR(S) found:")
    for e in errors:
        print("  •", e)
    raise RuntimeError("Preprocessing validation failed — see errors above.")
else:
    print("✓ All validation checks passed.")

if warnings:
    print(f"\n⚠ {len(warnings)} WARNING(S):")
    for w in warnings:
        print("  •", w)

✓ Wave coverage OK: [55, 56, 57]
✓ AFF_FEAR range OK after reversal: [1, 7]
✓ AFF_WORRY range OK after reversal: [1, 7]
✓ trust_stratum values OK: only 'low', 'high', NaN
✓ All 6 instance files present on disk


,file,rows_on_disk,n_cc_in_index,total_missing_node_vals,consistent
0,cosmo_nodes_wave55_trusthigh.csv,309,309,0,True
1,cosmo_nodes_wave55_trustlow.csv,376,376,0,True
2,cosmo_nodes_wave56_trusthigh.csv,351,351,0,True
3,cosmo_nodes_wave56_trustlow.csv,383,383,0,True
4,cosmo_nodes_wave57_trusthigh.csv,407,407,0,True
5,cosmo_nodes_wave57_trustlow.csv,336,336,0,True


✓ All instance files are complete-case and consistent with the index

✓ All validation checks passed.
